> **Teacher copy: all answers filled in.** Built from `notebooks/` by `solutions/make_solutions.py`; don't edit by hand, and don't share with learners.

# Notebook 3: Taking it home

**Framework:** Unsloth · **Model:** `unsloth/Qwen3.5-0.8B` · **Time:** ~25 minutes · **GPU:** free Colab T4

This notebook answers two questions: how do I **keep** what I've made, and how do I do this with **my own data**? It's also meant as a reference to come back to after the workshop.

**What you'll do**
1. Reload your pirate adapter from Google Drive, and export the model to a file you can run on your own laptop.
2. Run a template that fine-tunes the model on a CSV file. In the workshop it uses a sample dataset; at home you point it at your own.
3. Go through a checklist of common pitfalls.
4. See where to go next: bigger computers, bigger models, and how other researchers have used fine-tuning.
5. Decide which approach suits some research tasks, and plan one of your own.

**Learning outcomes covered:** 1 (prompting vs RAG vs fine-tuning), 2 (preparing data), 3 (LoRA), 4 (evaluation).

### Before you start: turn on the GPU
In Colab, go to **Runtime → Change runtime type**, choose **T4 GPU**, and click **Save**.

## 0. Setup

The same install as Notebook 1. **Run it now**: it takes a few minutes.

In [ ]:
%%capture
!pip install "unsloth==2026.11.3" "unsloth_zoo==2026.11.3"
!unsloth install-kernels

Connect Google Drive, so this notebook can read what you saved in Notebooks 1 and 2 and keep what you make here. Colab asks you to allow access.

Everything from the workshop lives in one folder in your Drive: `fine-tuning-workshop`.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")  # asks permission, then shows your Drive at /content/drive

DRIVE_DIR = "/content/drive/MyDrive/fine-tuning-workshop"
!mkdir -p {DRIVE_DIR}  # creates the folder if it isn't there yet
!ls {DRIVE_DIR}  # lists what you saved in Notebooks 1 and 2

### Helpers from Notebook 1

*Hidden code: run it.* These are the `chat` and `ask_many` functions from Notebook 1, unchanged.

In [ ]:
# @title Helpers from Notebook 1: `chat` and `ask_many` (double-click to see the code)
import torch
from unsloth import FastLanguageModel

SEED = 3407
MODEL_NAME = "unsloth/Qwen3.5-0.8B"


def chat(question, system_prompt=None, max_new_tokens=200):
    # 1. Build the conversation
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})  # sets the model's behaviour
    messages.append({"role": "user", "content": question})

    # 2. Apply the chat template, then turn the text into tokens on the GPU
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)  # one string, ending where the reply starts
    inputs = tokenizer(text=prompt, return_tensors="pt").to("cuda")

    # 3. Generate a reply
    torch.manual_seed(SEED)  # same reply each time
    output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                            temperature=0.7, do_sample=True)  # higher temperature = more random

    # 4. Keep only the new tokens (the reply), and turn them back into text
    reply_tokens = output[0][inputs["input_ids"].shape[1]:]  # skip the prompt, keep the reply
    return tokenizer.decode(reply_tokens, skip_special_tokens=True).strip()


def ask_many(model, tokenizer, questions, system_prompt=None, max_new_tokens=200, batch_size=16):
    replies = []
    for start in range(0, len(questions), batch_size):  # groups of batch_size questions
        batch = questions[start:start + batch_size]

        prompts = []
        for q in batch:
            messages = [{"role": "system", "content": system_prompt}] if system_prompt else []
            messages.append({"role": "user", "content": q})
            prompts.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=False))   # some Qwen models "think" out loud first unless told not to

        # Questions have different lengths, so the shorter ones are padded with filler.
        # Padding goes on the left, so every prompt ends exactly where its reply begins.
        inputs = tokenizer(text=prompts, return_tensors="pt",
                           padding=True, padding_side="left").to("cuda")

        torch.manual_seed(SEED)  # repeatable results
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                temperature=0.7, do_sample=True)
        reply_tokens = output[:, inputs["input_ids"].shape[1]:]  # keep only the replies
        replies += tokenizer.batch_decode(reply_tokens, skip_special_tokens=True)

    return [r.strip() for r in replies]

## 1. Reload and export your pirate model

Colab deletes everything when the session ends, which is why section 7 of Notebook 1 saved your adapter to Drive. Here you load it back, check it still talks like a pirate, and export it to run on your own computer.

### 1a. Load the base model and attach your adapter

Your adapter only stores the small LoRA matrices, plus the name of the base model it belongs to. Unsloth reads that name, downloads the base model, and attaches the adapter.

If your adapter isn't in Drive (for example, your training didn't finish), the cell downloads a copy we trained before the workshop from this workshop's GitHub repo.

In [ ]:
PIRATE_LORA_DIR = f"{DRIVE_DIR}/pirate_lora"
FALLBACK_URL = "https://raw.githubusercontent.com/UoA-eResearch/fine-tuning_workshop/main/adapters/pirate_lora.zip"   # the adapter we trained before the workshop

if os.path.exists(f"{PIRATE_LORA_DIR}/adapter_config.json"):  # did you save an adapter in Notebook 1?
    adapter_dir = PIRATE_LORA_DIR
    print("Using your adapter from Drive")
else:
    adapter_dir = "/content/pirate_lora_workshop"
    !wget -q {FALLBACK_URL} -O pirate_lora.zip  # downloads the zip from GitHub
    !unzip -q -o -j pirate_lora.zip -d {adapter_dir}  # -j drops the folders inside the zip
    print("No adapter in your Drive, so using the workshop's copy")

model, tokenizer = FastLanguageModel.from_pretrained(
    adapter_dir,  # loads the base model and attaches the adapter
    max_seq_length=1024,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)  # faster generation mode

Ask it a couple of questions with no system prompt. The answers should be in pirate speak, as at the end of Notebook 1.

In [ ]:
for q in ["What is photosynthesis?", "Write a two-line abstract for a paper on soil bacteria."]:
    print(f"Q: {q}\nA: {chat(q)}\n" + "-" * 60)

You could also push the adapter to the Hugging Face Hub to share it. We don't do that here: anything on a public hub is public, so check your data and ethics approvals first.

### 1b. Export to GGUF

**GGUF** is a single-file model format used by tools that run models on ordinary laptops, such as **Ollama** and **llama.cpp**. Exporting takes two steps: merge the adapter into the base model, then convert the merged model to one GGUF file.

We also **quantise** it: store the weights with fewer bits. `q8_0` uses 8 bits instead of 16, so the file is about 0.8 GB instead of 1.6 GB, with almost no loss in quality.

First, merge the adapter into the base weights and save the full model.

In [ ]:
model.save_pretrained_merged("pirate_merged", tokenizer)  # full 16-bit model, ~1.7 GB

Convert the merged model with **llama.cpp**'s conversion script. This needs no compiling, so it takes about a minute.

- We use a fixed version of llama.cpp from 12 November 2026. Later versions store Qwen3.5's layers in a newer layout that current Ollama can't load yet (`missing tensor 'blk.0.ssm_beta.weight'`).

In [ ]:
LLAMA_CPP_VERSION = "0fb8f157702702f33f4e018de8701b2569a1bc28"  # 12 Nov 2026, last version Ollama can load for Qwen3.5
!wget -q https://github.com/ggml-org/llama.cpp/archive/{LLAMA_CPP_VERSION}.tar.gz -O llama_cpp.tar.gz
!mkdir -p llama.cpp && tar -xzf llama_cpp.tar.gz -C llama.cpp --strip-components=1
!pip install -q ./llama.cpp/gguf-py
!python llama.cpp/convert_hf_to_gguf.py pirate_merged --outtype q8_0 --outfile pirate-qwen3.5-0.8b-Q8_0.gguf

Copy the GGUF file to Drive, and write a **Modelfile** next to it. A Modelfile tells Ollama which model file to use and which settings to run it with.

In [ ]:
import shutil

GGUF_DIR = f"{DRIVE_DIR}/pirate_gguf"
GGUF_NAME = "pirate-qwen3.5-0.8b-Q8_0.gguf"
os.makedirs(GGUF_DIR, exist_ok=True)  # creates the folder if needed
shutil.copy(GGUF_NAME, f"{GGUF_DIR}/{GGUF_NAME}")

# The Modelfile tells Ollama which file to load and how to run it
with open(f"{GGUF_DIR}/Modelfile", "w") as f:
    f.write(f"FROM ./{GGUF_NAME}\n")
    f.write("PARAMETER temperature 0.7\n")

!ls -lh {GGUF_DIR}  # -h shows sizes in MB/GB

### 1c. Run it on your own computer with Ollama

*The presenter now shows this on their laptop with the same file.*

To do it yourself afterwards:

1. Install **Ollama** from [ollama.com](https://ollama.com), or update it if you already have it.
2. In Google Drive, download the `fine-tuning-workshop/pirate_gguf` folder (the `.gguf` file and the `Modelfile`), and unzip it if Drive zips it. Keep both files in the same folder.
3. Open a terminal in that folder and run:
   ```bash
   ollama create pirate -f Modelfile
   ollama run pirate --think=false
   ```
4. Type a question. Type `/bye` to quit.

- `--think=false` turns off Qwen3.5's **thinking mode**, where the model writes out its reasoning before it answers. Ollama turns it on by default, but our model was trained without it. In a chat that's already open, type `/set nothink`.
- `ollama create` copies the model into Ollama, so afterwards `ollama run pirate --think=false` works from any folder.

The model now runs entirely on your computer, with no internet connection needed and no data leaving your machine.

**llama.cpp** is the lower-level option that Ollama is built on: `llama-cli -m pirate-qwen3.5-0.8b-Q8_0.gguf -cnv`.

Free the GPU before section 2.

In [ ]:
import gc

del model, tokenizer  # removes the pirate model
gc.collect()  # clears up anything still in memory
torch.cuda.empty_cache()  # frees the GPU memory it used
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 2. Template: fine-tune on your own data

This is the Notebook 1 workflow, driven by a few settings instead of written for one dataset. You give it a **CSV file** with an input column and a target column. The model learns to read the input and write the target.

The target can be anything you can write as text: a label ("Economics"), an extracted value ("12 mg"), a summary, or a rewritten sentence. A label is just a very short reply, so the same code works for all of them.

For DNA, protein or other non-text sequences, follow Notebook 2 (Transformers + PEFT) instead.

**The sample dataset** is 400 arXiv abstracts, each with its field: Astrophysics, Quantitative biology, Economics or Mathematics (100 of each). arXiv's abstracts are released under CC0 (public domain). See [`data/arxiv_abstracts_README.md`](https://github.com/UoA-eResearch/fine-tuning_workshop/blob/main/data/arxiv_abstracts_README.md) for how it was made.

**In the workshop:** run 2a to 2d. Sections 2e to 2i train and evaluate the model, so run them at home on your own data. They take about 10 minutes on a T4.

### 2a. Edit me

These are the only settings you need to change for your own data.
- `CSV_PATH`: a CSV file, either a URL or a path in your Drive (for example `f"{DRIVE_DIR}/my_data.csv"`; upload the file to the `fine-tuning-workshop` folder first).
- `INPUT_COL`: the column the model reads.
- `TARGET_COL`: the column the model should learn to write.
- `SYSTEM_PROMPT`: an instruction given with every input, or `None`. If your targets come from a fixed list, name the options here.
- `MAX_NEW_TOKENS`: the longest reply the model may write, in **tokens** (word pieces, roughly ¾ of a word). Keep it small for short targets; raise it for summaries or longer text.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/UoA-eResearch/fine-tuning_workshop/main/data"

CSV_PATH = f"{DATA_URL}/arxiv_abstracts.csv"  # a URL, or a file path in Drive
INPUT_COL = "abstract"  # the column the model reads
TARGET_COL = "field"  # the column it learns to write
# Optional: set to None for no system prompt
SYSTEM_PROMPT = ("Which field is this research abstract from? Reply with one of: "
                 "Astrophysics, Quantitative biology, Economics, Mathematics.")
MAX_NEW_TOKENS = 20     # try 300 for longer text

### 2b. Check the data

Most fine-tuning problems are data problems. This cell removes empty rows and duplicate inputs, then shows the most common targets.

If your targets are labels, check they're balanced: if 90% of rows have one label, a model that always says that label is right 90% of the time.

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)[[INPUT_COL, TARGET_COL]]  # keep only the two columns we need
print(f"{len(df)} rows")

df = df.dropna().drop_duplicates(subset=INPUT_COL).astype(str).reset_index(drop=True)  # renumber rows from 0
print(f"{len(df)} rows after removing empty rows and duplicate inputs")

print("\nMost common targets:")
print(df[TARGET_COL].value_counts().head(10).to_string())  # the 10 most common targets and their counts

### 2c. Convert to chat format

Each row becomes the same `{"messages": [...]}` format as the pirate dataset: the input is the user's turn, and the target is the assistant's reply. If you set a system prompt, it goes first.

In [ ]:
import json

def to_messages(input_text, target_text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] if SYSTEM_PROMPT else []
    messages.append({"role": "user", "content": input_text})  # the input
    messages.append({"role": "assistant", "content": target_text})  # the reply we want
    return messages

df["messages"] = [to_messages(i, t) for i, t in zip(df[INPUT_COL], df[TARGET_COL])]  # one conversation per row
print(json.dumps({"messages": df["messages"][0]}, indent=2))  # the first example, in the pirate dataset's format

### 2d. Train, validation and test sets

The same three sets as Notebook 1: 70% train, 10% validation, 20% test. The fixed seed gives the same split every time, so the test set stays the same and your scores stay comparable.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)  # 20% for the test set
train_df, val_df = train_test_split(train_df, test_size=0.125, random_state=SEED)   # see the question below

print(f"Training: {len(train_df)}   Validation: {len(val_df)}   Test: {len(test_df)}")

**Question:** we want 10% of all rows for validation. Why is `test_size` 0.125 in the second split, not 0.1?

**Answer (teacher copy):**

The second split only sees the 80% of rows left after the test set is taken out. 0.125 × 80% = 10% of all rows. With 0.1 you'd get 8%.

*In the workshop, stop here. The rest of section 2 trains the model: run it at home on your own data.*

---

### 2e. Baseline: the model before training

Load the base model and run it on the test set, with your system prompt. This is the **prompting baseline** from Notebook 1: if prompting alone is good enough, you don't need to fine-tune.

The score is **exact match**: the share of replies that equal the target, ignoring capitals and a trailing full stop. For labels and short values this is accuracy. For longer text it stays near 0 even when the replies are good, so read the replies instead.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=1024, load_in_4bit=False)  # the base model, no adapter
FastLanguageModel.for_inference(model)

test_inputs = test_df[INPUT_COL].tolist()
test_targets = test_df[TARGET_COL].tolist()

def exact_match(replies):
    # True where the reply matches the target, ignoring case and a trailing full stop
    same = [r.strip(" .").lower() == t.strip(" .").lower() for r, t in zip(replies, test_targets)]
    return sum(same) / len(same)  # the fraction that match

before = ask_many(model, tokenizer, test_inputs, system_prompt=SYSTEM_PROMPT, max_new_tokens=MAX_NEW_TOKENS)
print(f"Exact match before training: {exact_match(before):.0%}")
pd.set_option("display.max_colwidth", 300)  # show up to 300 characters per cell
display(pd.DataFrame({"input": test_inputs, "target": test_targets, "reply": before}).head(5))  # the first 5 test rows

### 2f. Train

The same LoRA set-up and trainer as Notebook 1 (sections 3 and 5). Three settings are worth changing:
- `MAX_STEPS`: how many training steps. Raise it if the validation loss is still falling at the end; lower it if the validation loss starts rising (overfitting).
- `LEARNING_RATE`: how big each update is. Lower it (e.g. `1e-4`) if the loss jumps around or the model starts producing nonsense.
- `LORA_RANK`: the size of the adapters. Try 8 for small or simple tasks, 32 if the model isn't learning enough.

With 280 training rows and 8 rows per step, 60 steps is about 1.7 passes through the data.

In [ ]:
MAX_STEPS = 60          # training steps; try 30 or 120
LEARNING_RATE = 2e-4    # step size; try 1e-4 or 5e-5
LORA_RANK = 16          # adapter size; try 8 or 32

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,  # skip the image encoder
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,  # same LoRA settings as Notebook 1
    r=LORA_RANK,
    lora_alpha=LORA_RANK,  # scaling factor, set equal to r
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves GPU memory
    random_state=SEED,
)

def to_dataset(part):
    texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in part["messages"]]  # each chat as one string
    return Dataset.from_dict({"text": texts})  # a dataset with a "text" column

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=to_dataset(train_df),  # the data the model learns from
    eval_dataset=to_dataset(val_df),  # checks for overfitting during training
    args=SFTConfig(
        per_device_train_batch_size=8,  # lower to 4 or 2 if you run out of GPU memory
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        max_length=1024,          # longer examples are cut off
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=20,  # validation loss every 20 steps
        seed=SEED,
        report_to="none",  # no online tracking service
    ),
)
trainer = train_on_responses_only(  # learn only from the assistant's replies
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print(f"Training took {stats.metrics['train_runtime']:.0f} seconds")

The training loss should fall. If the validation loss starts rising while the training loss keeps falling, the model is overfitting.

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history  # everything the trainer logged
train = [(l["step"], l["loss"]) for l in logs if "loss" in l]  # (step, training loss)
val = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]  # (step, validation loss)
plt.plot(*zip(*train), label="training loss")
if val:
    plt.plot(*zip(*val), "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

### 2g. Evaluate: before vs after

The same test set and system prompt as the baseline. Compare the exact-match scores, then read the replies side by side.

In [ ]:
FastLanguageModel.for_inference(model)  # back to generation mode
after = ask_many(model, tokenizer, test_inputs, system_prompt=SYSTEM_PROMPT, max_new_tokens=MAX_NEW_TOKENS)  # same test set and settings as the baseline

print(f"Exact match   before: {exact_match(before):.0%}   after: {exact_match(after):.0%}")
display(pd.DataFrame({"input": test_inputs, "target": test_targets,
                      "before": before, "after": after}).head(10))

**Think about it:**
- Did the replies change **format** (e.g. exactly one label, no extra words) as well as content? Fine-tuning is good at teaching format.
- Where it's wrong, would a human expert get it wrong too? Papers that sit between two fields (for example, mathematical biology) are hard for everyone.
- For longer text, exact match can't tell good replies from bad ones. Read a sample, or use an LLM as a judge (Notebook 1).

> **Answer (teacher copy):**
>
> - **Format:** before training, expect replies like "This abstract is from the field of Astrophysics because..." that fail exact match even when the field is right. After training, expect one label and nothing else. Much of the before/after gain in exact match can come from format, not understanding. That's worth pointing out: fine-tuning is very good at teaching format.
> - **Confusions:** look for Quantitative biology vs Mathematics, and Economics vs Mathematics (theory or econometrics papers). Astrophysics is usually the easiest. Experts would also disagree on papers that sit between fields, so some errors reflect the labels, not the model.
> - **Longer text:** exact match is close to 0 even for good summaries, because any different word counts as wrong. Read a sample, or use an LLM judge with a clear rubric.

### 2h. Apply the model to new rows

This is how you'd use the model in research: run it over new data and get a CSV back. `NEW_CSV_PATH` needs the same `INPUT_COL` as your training data. The results are saved to Drive, ready to open in Excel, R or pandas.

In [ ]:
NEW_CSV_PATH = f"{DATA_URL}/arxiv_abstracts_new.csv"  # new rows, with no targets
OUT_DIR = f"{DRIVE_DIR}/your_data"

new_df = pd.read_csv(NEW_CSV_PATH)
# Add the model's replies as a new column
new_df[f"predicted_{TARGET_COL}"] = ask_many(model, tokenizer, new_df[INPUT_COL].astype(str).tolist(),
                                             system_prompt=SYSTEM_PROMPT, max_new_tokens=MAX_NEW_TOKENS)

os.makedirs(OUT_DIR, exist_ok=True)
new_df.to_csv(f"{OUT_DIR}/predictions.csv", index=False)  # index=False leaves out the row numbers
print(f"Saved {len(new_df)} predictions to {OUT_DIR}/predictions.csv")
new_df.head(10)

### 2i. Save the adapter

Saved to Drive, next to your predictions. To use it later, load it as in section 1a.

In [ ]:
model.save_pretrained(f"{OUT_DIR}/lora")  # saves only the small LoRA adapter
tokenizer.save_pretrained(f"{OUT_DIR}/lora")  # saved alongside, for reloading
!ls -lh {OUT_DIR}/lora

## 3. Pitfalls checklist

Check these before you trust (or publish) a fine-tuned model.

| Pitfall | How to spot it | What to do |
|---|---|---|
| **Data leakage**: test examples, or near-copies of them, are in the training data | Test scores look too good to be true | Remove duplicates before splitting (2b). Split by group when rows are related: same patient, same paper, same chromosome (as in Notebook 2's test set) |
| **Overfitting**: the model memorises the training data | Training loss falls while validation loss rises | Fewer steps, more data, or a smaller rank |
| **Catastrophic forgetting**: the model loses things it could do before | General questions now get wrong answers (Notebook 1, section 6c) | Fewer steps, a lower learning rate, or mix some general examples into the training data |
| **No baseline** | You can't say whether fine-tuning helped | Always measure the untrained model, a prompted model, and a simple guess such as the most common label |
| **Results you can't reproduce** | A rerun gives different numbers, or you can't remember the settings | Record the model name and version, seed, settings, package versions and data version. Keep the test file fixed |

**Quiz:** which pitfall is each researcher running into?

1. A model that classifies patient notes scores 98% on its test set. Each patient has several notes, and the notes were split into training and test sets at random.
2. After 500 training steps the training loss is still falling, but the validation loss has been rising since step 200.
3. After fine-tuning to extract drug doses, the model gets simple general questions wrong that it used to get right.
4. A fine-tuned classifier reaches 85% accuracy, but 85% of the test rows have the same label.
5. A colleague reruns your notebook six months later and gets different numbers.

**Answer (teacher copy):**

1. **Data leakage:** notes from the same patient are in both sets. Split by patient.
2. **Overfitting:** stop around step 200, or add data.
3. **Catastrophic forgetting.**
4. **No baseline:** always guessing the most common label also scores 85%.
5. **Results you can't reproduce:** record versions, seeds, settings and data.

## 4. Where to go next

### More computing power

- **University of Auckland:** the [advanced compute page](https://www.auckland.ac.nz/en/research/research-resources/research-ai-software-computing/advanced-compute.html) lists the options. The most relevant is the **[Local AI Service (LAIS)](https://www.auckland.ac.nz/en/research/research-resources/research-ai-software-computing/advanced-compute/local-ai-service.html)**:
  - Dedicated Dell Pro Max GB10 machines with 128 GB of memory (models up to ~200 billion parameters), or 256 GB with two units (~400 billion).
  - Free. Your data and models stay on University premises, and machines are wiped between groups.
  - SSH access with CUDA, JupyterLab and Docker. Book from one week, extendable to four.
  - Staff and doctoral candidates request access directly; other students go through their supervisor. Contact researchdata@auckland.ac.nz.
  - This workshop is adapted from [ahoy_training](https://github.com/UoA-eResearch/ahoy_training), which was built for the same GB10 hardware, so its full-size demos are a natural next step.
- **Sensitive data** (for example, clinical notes): use the **Secure Research Environment (SRE)**, on the same advanced compute page.
- **Off campus:** free Colab has daily GPU limits; Colab Pro and Kaggle notebooks give more GPU time.

### What changes with bigger models

- **Memory:** roughly 2 bytes per parameter in 16-bit, about 0.5 bytes in 4-bit. An 8B model needs ~16 GB in 16-bit, more than a T4 has, so **QLoRA** (LoRA on a 4-bit model) becomes necessary.
- **Speed:** training and generation are slower, and very large models may need several GPUs. ahoy_training fine-tunes a 72B model with QLoRA, as a concrete example.
- **Results:** bigger models usually do better, and often need fewer examples.

### How other researchers have used fine-tuning

**Discussed in the session**

1. **Chemistry: molecular properties** ([Jablonka et al., 2024, *Nature Machine Intelligence*](https://doi.org/10.1038/s42256-023-00788-1)). GPT-3 fine-tuned to answer chemistry and materials questions in plain language. It matched or beat conventional machine learning, especially with little data, and could suggest new molecules by asking the questions in reverse.
2. **Medicine: extracting data from clinical reports** ([*Scientific Reports*, 2025](https://doi.org/10.1038/s41598-025-28767-z)). Llama-3.1-8B fine-tuned with LoRA did as well as a second human annotator on four datasets, using desktop hardware and 100 or fewer training reports.
3. **Radiology: a teacher model labels the data** ([Chen et al., 2024, BURExtract-Llama](https://arxiv.org/abs/2408.11334); workshop paper). GPT-4 labelled a small dataset, then Llama was fine-tuned on it and matched GPT-4 at lower cost and with better privacy.
4. **Linguistics and data sovereignty: te reo Māori speech recognition** ([Te Hiku Media / Papa Reo](https://tehiku.nz/te-hiku-tech/kaituhi-videos/)). Whisper fine-tuned on data under their Kaitiakitanga Licence performed much better, but they chose not to use it in production because of ethical concerns and the potential harm to Indigenous languages.

**More to read**

5. **Materials science: from papers to databases** ([Dagdelen et al., 2024, *Nature Communications*](https://doi.org/10.1038/s41467-024-45563-x)). GPT-3 and Llama-2 fine-tuned to extract structured records (sentences or JSON) from papers, with around 500 examples.
6. **Political science: annotating text** ([Alizadeh et al., 2024, *Journal of Computational Social Science*](https://arxiv.org/abs/2307.02179)). Open-source models fine-tuned for stance, topic and relevance in news and tweets matched or beat GPT-3.5 and GPT-4 without fine-tuning, and beat few-shot prompting. The authors share a notebook.
7. **Systematic reviews: screening titles and abstracts** ([Yamoah et al., 2026](https://arxiv.org/abs/2603.24767); preprint). A 1.2B model fine-tuned on more than 8,500 human-rated titles and abstracts improved a lot and missed few relevant papers, so it could act as a second screener.
8. **Humanities: correcting OCR errors in historical newspapers** (Thomas et al., 2024, LT4HALA workshop). Instruction-tuned Llama 2 compared with fine-tuned BART for finding and fixing OCR errors in 19th-century British newspapers (BLN600).
9. **Education: essay scoring.** Human-AI Collaborative Essay Scoring fine-tuned Llama-3-8B (4-bit) with Unsloth and LoRA at rank 16 and alpha 16, almost the settings used here. [AiAWE (2026)](https://arxiv.org/abs/2606.12801) scores argumentative essays with a LoRA-adapted Gemma-3-27B.
10. **Astronomy: adapting to a whole field** ([Nguyen et al., 2023, AstroLLaMA](https://arxiv.org/abs/2309.06126); preprint). LLaMA-2-7B trained further on more than 300,000 astronomy abstracts.

### Other ways to train

- **DPO** (Direct Preference Optimisation): instead of example answers, you give pairs of answers marked better and worse, and the model learns to prefer the better kind.
- **GRPO** (Group Relative Policy Optimisation): the model generates several answers, a scoring function you write rates them, and the model learns from the best. Used to train reasoning on tasks with checkable answers, such as maths.

Unsloth supports both.

### Links

- [Unsloth documentation](https://docs.unsloth.ai) and [Unsloth notebooks](https://github.com/unslothai/notebooks) (many models and tasks, ready for Colab)
- [Hugging Face LLM course](https://huggingface.co/learn/llm-course)
- [This workshop's materials](https://github.com/UoA-eResearch/fine-tuning_workshop)

## 5. Which approach would you use?

For each research task, would you start with **prompting**, **RAG** (retrieval-augmented generation: the model looks things up in your documents before it answers) or **fine-tuning**? Some have more than one good answer.

1. Answer questions about your lab's protocols, which change every few months.
2. Sort 10,000 free-text survey responses into 5 categories you've defined. Prompting gets about 70% right, and you have 500 hand-labelled examples.
3. Summarise a few papers a week for your own reading.
4. Turn notes into a strict JSON format for a data pipeline, with a small local model that keeps breaking the format when prompted.
5. Extract drug doses from clinical notes that can't leave the University's systems.

**Answer (teacher copy):**

1. **RAG.** The facts change, and fine-tuning is a poor way to add facts. Update the documents, not the model.
2. **Fine-tuning.** A fixed task, labelled examples, a prompting baseline that isn't good enough, and many items to process.
3. **Prompting.** Small scale, no training data, and probably good enough already.
4. **Fine-tuning.** Format and behaviour are what fine-tuning teaches best (like the pirate style).
5. **Prompting a local model first, then fine-tuning one** (for example on LAIS or in the SRE) if that isn't good enough. The notes can't be sent to an outside service, and that includes a teacher model behind an API.

### Plan it for your own research

Double-click this cell and fill in the right-hand column.

| | Your answer |
|---|---|
| **Task** | |
| **Input** (what the model reads) | |
| **Target** (what it should write) | |
| **Prompting, RAG or fine-tuning, and why** | |
| **Baseline to beat** | |
| **How you'll measure success** | |
| **Examples you have, or could make** | |
| **Can the data leave the University?** | |

> **Answer (teacher copy):**
>
> There's no single answer; ask two or three people to share. A worked example using the sample data:
>
> | | Example |
> |---|---|
> | **Task** | Sort new arXiv preprints into our four fields for a reading list |
> | **Input** | The abstract |
> | **Target** | One field name |
> | **Prompting, RAG or fine-tuning, and why** | Fine-tuning: fixed labels, a format the prompted model breaks, and thousands of abstracts to sort |
> | **Baseline to beat** | The prompted model's exact match (2e), and always guessing the most common field (25% here) |
> | **How you'll measure success** | Exact match (accuracy) on the 80-row test set, plus reading the mistakes |
> | **Examples you have, or could make** | 400 labelled abstracts; more from arXiv's metadata |
> | **Can the data leave the University?** | Yes, it's public (CC0), so an API teacher model would be allowed |
>
> Things to listen for: a target that's actually new facts (that's RAG), no baseline, no test set, and sensitive data heading for an outside API.

## Recap

- **Keep your work:** save adapters to Drive; export to GGUF to run the model on your own computer with Ollama.
- **Your own data:** a CSV with an input and a target column is enough to start. Check it, split it, measure a baseline, then train.
- **Watch for** leakage, overfitting, forgetting and missing baselines, and record what you did.
- **Next steps:** the University's Local AI Service for bigger models, and the research examples above for ideas.